# Verificación — Spark lee tu datalake

**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Equipo:** Mateo Sanz Medina, Samuel Arango, Nathalia Cardoza

**Fecha:** 2026-08-13

## Objetivo

Cerrar el Lab 1a confirmando que tu clúster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

BUCKET = "st1630-msanzm-2026"

ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

print("Filas leídas:", df_parquet.count())


root
 |-- id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- total: double (nullable = true)

+--------------------+----------+----------+--------+-----------+------------+--------+---------------+------+  
|id                  |fecha     |cliente_id|region  |categoria  |producto    |cantidad|precio_unitario|total |  
+--------------------+----------+----------+--------+-----------+------------+--------+---------------+------+  
|e7a1029c-1204-44b2-a|2025-01-15|CLI-0492  |Bogotá  |Electrónica|Laptop      |2       |1250.0         |2500.0|  
|9c831f23-88ab-4091-b|2025-01-16|CLI-0112  |Medellín|Hogar      |Cafetera    |1       |85.5           |85.5  |  
|3b56a908-1123-45a1-c|2025-01-17|CLI-0841  |Bogotá  |Electrónica|Monitor

In [2]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 0.158 s (6 filas de resultado)
CSV: 1.398 s (6 filas de resultado)

Ratio CSV / Parquet: 8.85x


## Análisis — respuestas del laboratorio

### a) Tamaño en disco

`prueba_parquet.parquet` pesa **185.0 KB** (189,416 bytes) frente a `prueba_csv.csv` que pesa **798.3 KB** (817,409 bytes). Parquet logra una reducción de tamaño de **4.32x** gracias a la compresión columnar por diccionario y algoritmos como Snappy.

### b) Tiempo de la consulta

La consulta selectiva (`region == "Bogotá"` y `categoria == "Electrónica"`, agrupada por producto) tomó:
- **Parquet:** `0.158 s`
- **CSV:** `1.398 s`

### c) Ratio de mejora

El ratio de mejora obtenido fue de **8.85x** a favor de Parquet. Coincide plenamente con el orden de magnitud visto en clase (~9x). Esto se debe a que Spark aprovecha el *column pruning* (solo lee las columnas necesarias) y *predicate pushdown* (descarta bloques leyendo metadatos min/max), mientras que en CSV debe escanear y parsear el texto de todo el archivo línea por línea.

### d) Conexión con el Teorema CAP

AWS S3 con replicación entre múltiples zonas de disponibilidad es una decisión **CP** (Consistencia y Partición) porque garantiza **Consistencia de Lectura tras Escritura** (*Strong Read-After-Write Consistency*). S3 no confirma una operación de escritura hasta que los datos están durablemente replicados en múltiples AZs. Ante una partición de red entre AZs, S3 prefiere retrasar o rechazar la confirmación antes de permitir que un cliente lea datos desactualizados o inconsistentes.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

## Bitácora de delegación y trabajo en equipo

| Tarea | Integrante Responsable | ¿Delegado a agente? | Herramienta / Justificación |
|---|---|---|---|
| Setup S3 y Estructura Medallion | Samuel | No | Creación del bucket y prefijos ronze/, silver/, gold/. |
| Generación de datos sintéticos | Samuel | Sí | Ejecución de generar_datos.py con pandas y pyarrow. |
| Políticas IAM Mínimo Privilegio | Nathalia | No | Definición de recurso acotado a s3://st1630-msanzm-2026/*. |
| Análisis del Teorema CAP y CAPex | Nathalia | No | Justificación de S3 como sistema CP con Consistencia Fuerte. |
| Setup AWS CLI y KeyPair SSH | Mateo | Sí | Configuración de credenciales temporales y formato PEM RSA. |
| Aprovisionamiento EMR y PySpark | Mateo | Sí | Creación del clúster EMR j-0889427F9EQIW3P0IGI y ejecución Spark. |
| Captura Spark UI DAG (Exchange) | Mateo | No | Navegación en Spark History Server y captura del nodo Exchange. |
